In [ ]:
import threading
import mido
from mido import MidiFile
from IPython.display import display
import ipywidgets as widgets

class MidiPlayer:
    def __init__(self):
        self.playlist = []
        self.current_track = 0
        self.playing = False
        self.paused = False
        self.thread = None

    def add_to_playlist(self, midi_file):
        self.playlist.append(midi_file)

    def play(self):
        if not self.playing:
            self.playing = True
            self.thread = threading.Thread(target=self._play_midi)
            self.thread.start()

    def pause(self):
        self.paused = not self.paused

    def skip(self):
        self.current_track += 1
        if self.current_track >= len(self.playlist):
            self.current_track = 0

    def stop(self):
        self.playing = False
        if self.thread is not None:
            self.thread.join()

    def _play_midi(self):
        while self.playing:
            midi_file = self.playlist[self.current_track]
            for msg in MidiFile(midi_file):
                if not self.paused:
                    mido.sleep(msg.time)
                    if not msg.is_meta:
                        print(msg)
            self.current_track += 1
            if self.current_track >= len(self.playlist):
                self.current_track = 0

from utils.list_files import list_files

# Example usage
player = MidiPlayer()
for fn in list_files('output'):
    player.add_to_playlist(fn)

play_button = widgets.Button(description="Play")
pause_button = widgets.Button(description="Pause")
skip_button = widgets.Button(description="Skip")

play_button.on_click(lambda x: player.play())
pause_button.on_click(lambda x: player.pause())
skip_button.on_click(lambda x: player.skip())

display(play_button, pause_button, skip_button)